### Setup

In [1]:
import numpy as np
import choix
from scipy.optimize import minimize
import scipy.stats as stats
import matplotlib.pyplot as plt
import random
from matplotlib import colors
import pandas as pd
import seaborn as sns
import pickle
import os
import sys

sys.path.append(os.path.abspath('../../'))
from metrics import compute_acc, compute_weighted_acc
from opt_fair import *
from distribution_utils import safe_kendalltau, to_numpy

In [2]:
!nvidia-smi

Sat Jun 27 06:15:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:17:00.0 Off |                    0 |
| N/A   50C    P0             46W /  300W |       1MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print(device)

cuda


In [4]:
print(f"Current PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

Current PyTorch version: 2.4.0a0+f70bd71a48.nv24.06
CUDA available: True
CUDA version: 12.5


In [5]:
with open("data/FaceAgePC.pickle", 'rb') as handle:
    PC_faceage = pickle.load(handle)    
with open("data/FaceAgeDF.pickle", 'rb') as handle:
    df_faceage = pickle.load(handle)

In [6]:
df_faceage

,full_path,score,gender
0,nm1442940_rm3965098752_1996-10-3_2006.jpg,10,0.0
1,nm4832920_rm1781768448_2003-8-28_2013.jpg,10,0.0
2,nm0652089_rm860657920_1992-3-10_2002.jpg,10,0.0
3,nm0004917_rm1493730304_1969-5-12_1979.jpg,10,0.0
4,nm1113550_rm1332711936_1996-4-14_2006.jpg,10,0.0
...,...,...,...
9145,475367_1941-08-03_2011.jpg,70,1.0
9146,304085_1919-07-07_1989.jpg,70,1.0
9147,nm0001627_rm4164078592_1927-2-20_1997.jpg,70,1.0
9148,nm0000024_rm1715129344_1904-4-14_1974.jpg,70,1.0


In [7]:
import opt_fair
all_pc_faceage  = opt_fair._pc_without_reviewers(PC_faceage)

size = len(df_faceage)
print(size)

9150


In [8]:
print(len(all_pc_faceage))

250249


In [9]:
gt_scores = to_numpy(df_faceage['score'].tolist())

In [10]:
max_iter=30000

In [11]:
lr = 0.001

In [12]:
tol = 1e-6

### HTCV

In [13]:
import random
from htcv_m import *

In [14]:
import time
Grad_accs, Grad_waccs, Grad_taus = [], [], []
gradem_time = []
for sd in range(1):
    start = time.time()
    grad_em = HTCVWrapper(PC_faceage, lr, sd, device, max_iter=max_iter)
    r_est, beta_est = grad_em.run_algorithm()
    end = time.time()
    gradem_time.append(end-start)

    r_est_np = to_numpy(r_est)
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    grad_acc = compute_acc(df_faceage, 1*r_est_np, device)
    grad_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    grad_tau = safe_kendalltau(r_est_np, gt_scores)
    
    Grad_accs.append(grad_acc)    
    Grad_waccs.append(grad_wacc)    
    Grad_taus.append(grad_tau)

 18%|█████████████▋                                                               | 5328/30000 [00:38<02:58, 137.93it/s]



 HTCV: Converged at epoch 5328 | Loss: 0.253280


In [15]:
print(f"{np.mean(gradem_time)} +- {np.std(gradem_time)}")

40.68747639656067 +- 0.0


In [16]:
print(f"GradEM -- Accuracy: {np.mean(Grad_accs)} ± {np.std(Grad_accs)}, Weighted Accuracy: {np.mean(Grad_waccs)} ± {np.std(Grad_waccs)}, Kendall's Tau: {np.mean(Grad_taus)} ± {np.std(Grad_taus)}")

GradEM -- Accuracy: 0.791709840297699 ± 0.0, Weighted Accuracy: 0.8732934594154358 ± 0.0, Kendall's Tau: 0.5786494276101926 ± 0.0


### HBTL

In [17]:
import random
from grad_em import *

In [18]:
import time
Grad_accs, Grad_waccs, Grad_taus = [], [], []
gradem_time = []
for sd in range(1):
    start = time.time()
    grad_em = GradientEMWrapper(PC_faceage, lr, sd, device, max_iter=max_iter)
    r_est, beta_est = grad_em.run_algorithm()
    end = time.time()
    gradem_time.append(end-start)

    r_est_np = to_numpy(r_est)
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    grad_acc = compute_acc(df_faceage, 1*r_est_np, device)
    grad_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    grad_tau = safe_kendalltau(r_est_np, gt_scores)
    
    Grad_accs.append(grad_acc)    
    Grad_waccs.append(grad_wacc)    
    Grad_taus.append(grad_tau)

 42%|███████████████████████████████▊                                            | 12573/30000 [01:27<02:01, 143.49it/s]

HBTL: Converged at epoch 12573 | Loss: 0.255266


In [19]:
print(f"{np.mean(gradem_time)} +- {np.std(gradem_time)}")

89.02212810516357 +- 0.0


In [20]:
print(f"GradEM -- Accuracy: {np.mean(Grad_accs)} ± {np.std(Grad_accs)}, Weighted Accuracy: {np.mean(Grad_waccs)} ± {np.std(Grad_waccs)}, Kendall's Tau: {np.mean(Grad_taus)} ± {np.std(Grad_taus)}")

GradEM -- Accuracy: 0.7920328378677368 ± 0.0, Weighted Accuracy: 0.8734515309333801 ± 0.0, Kendall's Tau: 0.5792902305596876 ± 0.0


### PG EM

In [21]:
import random
from pgem_initialize_beta_1 import EMWrapper, EMWrapper_3_0

In [22]:
PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
pgem_time = []
for sd in range(1):
    start = time.time()
    pg = EMWrapper(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    pgem_time.append(end-start)

    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda
cuda


  0%|                                                                               | 1/30000 [00:00<1:30:05,  5.55it/s]

Iter 000: Log-likelihood = -0.366355


  0%|▏                                                                               | 77/30000 [00:06<40:57, 12.17it/s]


Converged at iter 77, Log-likelihood change = 9.238720e-07


In [23]:
print(f"{np.mean(pgem_time)} +- {np.std(pgem_time)}")

7.518019437789917 +- 0.0


In [24]:
PGEM_accs

[0.7921259999275208]

In [25]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.7921259999275208 ± 0.0, Weighted Accuracy: 0.8736113905906677 ± 0.0, Kendall's Tau: 0.5794750337386451 ± 0.0


### Faster PGEM

In [26]:
import time

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []

for sd in range(1):
    start = time.time()
    pg = EMWrapper_3_0(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    print("Elapsed:", end - start, "seconds")
    
    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda


  0%|                                                                               | 17/30000 [00:00<03:06, 160.57it/s]

Iter 000: Log-likelihood = -0.367956


  0%|▏                                                                              | 62/30000 [00:00<03:16, 152.39it/s]


Converged at iter 62, ΔLL = 9.834766e-07
Elapsed: 1.492771863937378 seconds


In [27]:
PGEM_accs

[0.7920684814453125]

In [28]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.7920684814453125 ± 0.0, Weighted Accuracy: 0.8735842108726501 ± 0.0, Kendall's Tau: 0.5793608737593395 ± 0.0


### BT

In [13]:
%%time
bt_scores = opt_fair.opt_pairwise_gpu(size, all_pc_faceage, alpha=0, initial_params=None, max_iter=max_iter, tol=tol)

BT finished
CPU times: user 1.9 s, sys: 517 ms, total: 2.41 s
Wall time: 2.45 s


In [14]:
r_est_np = to_numpy(bt_scores)
bt_acc = compute_acc(df_faceage, 1*r_est_np, device)
bt_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
bt_tau = safe_kendalltau(r_est_np, gt_scores)

In [15]:
print(f"Simple BT -- Accuracy: {bt_acc}, Weighted Accuracy: {bt_wacc}, Kendall's Tau: {bt_tau} ")

Simple BT -- Accuracy: 0.7900682687759399, Weighted Accuracy: 0.8719562292098999, Kendall's Tau: 0.5753930665630511 


### BARP

In [16]:
classes = df_faceage['gender']
FaceAge = opt_fair.BARP(data = PC_faceage, penalty = 0, classes = classes, device=device)

In [17]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [18]:
%%time
annot_bt_temp, annot_bias =  opt_fair._alternate_optim_torch(size, num_reviewers, FaceAge, lr_x=lr, lr_y=lr, tol=tol, iters = max_iter)

 34%|█████████████████████████▌                                                  | 10089/30000 [00:35<01:10, 281.28it/s]

CPU times: user 34.8 s, sys: 1.2 s, total: 36 s
Wall time: 35.9 s


In [19]:
r_est_np = to_numpy(annot_bt_temp)
gt_scores = to_numpy(df_faceage['score'].tolist())
barp_acc = compute_acc(df_faceage, 1*r_est_np, device)
barp_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
barp_tau = safe_kendalltau(r_est_np, gt_scores)

In [20]:
print(f"BARP -- Accuracy: {barp_acc}, Weighted Accuracy: {barp_wacc}, Kendall's Tau: {barp_tau} ")

BARP -- Accuracy: 0.7908041477203369, Weighted Accuracy: 0.8726015090942383, Kendall's Tau: 0.5768528788223168 


### RC

In [21]:
%%time

rc_obj = RankCentrality(device)
A = rc_obj.matrix_of_comparisons(size, all_pc_faceage)
# print("A matrix done")
P = rc_obj.trans_prob(A)
# print("P matrix done")
pi = rc_obj.stationary_dist(P, max_iter=max_iter, tol=tol)
rank_centrality_scores = torch.log(pi).cpu().numpy()

RC converged at iteration 2
RC finished
CPU times: user 152 ms, sys: 112 ms, total: 264 ms
Wall time: 260 ms


In [22]:
r_est_np = to_numpy(rank_centrality_scores)
rc_acc = compute_acc(df_faceage, 1*r_est_np, device)
rc_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
rc_tau = safe_kendalltau(r_est_np, gt_scores)

In [23]:
print(f"RC -- Accuracy: {rc_acc}, Weighted Accuracy: {rc_wacc}, Kendall's Tau: {rc_tau} ")

RC -- Accuracy: 0.7804015278816223, Weighted Accuracy: 0.8644048571586609, Kendall's Tau: 0.5582073434385082 


### CrowdBT

In [40]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [41]:
print(device)
gt_scores = to_numpy(df_faceage['score'].tolist())

cuda


In [42]:
CrowdBT_accs, CrowdBT_waccs, CrowdBT_taus = [], [], []
K = num_reviewers
gt_df = df_faceage
crowdbt_time = []
for seed in range(1):
    try:
        start = time.time()
        crowdbt_test = opt_fair.CrowdBT_3_0(data=PC_faceage, penalty=0, device=device, random_seed=seed)
        crowdbt_scores, _ = crowdbt_test.alternate_optim(size, K, lr_x=lr, lr_y=lr, tol=tol, iters=max_iter)
        end = time.time()
        crowdbt_time.append(end-start)
        r_est_np = to_numpy(crowdbt_scores)
        gt_scores = to_numpy(df_faceage['score'].tolist())
        current_tau = safe_kendalltau(r_est_np, gt_scores)
        if current_tau < 0:
            r_est_np = -r_est_np
        crowdbt_acc = compute_acc(df_faceage, 1*r_est_np, device)
        crowdbt_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
        crowdbt_tau = safe_kendalltau(r_est_np, gt_scores)
        CrowdBT_accs.append(crowdbt_acc)
        CrowdBT_waccs.append(crowdbt_wacc)
        CrowdBT_taus.append(crowdbt_tau)
    except Exception as e:
        print(e)
        CrowdBT_accs.append(0.0)
        CrowdBT_waccs.append(0.0)
        CrowdBT_taus.append(0.0)

 75%|██████████████████████████████████████████████▊               | 22628/30000 [02:02<00:39, 184.97it/s, loss=6.41e+4]


In [43]:
print(f"{np.mean(crowdbt_time)} +- {np.std(crowdbt_time)}")

123.70051527023315 +- 0.0


In [44]:
print(f"CrowdBT -- Accuracy: {np.mean(CrowdBT_accs)} ± {np.std(CrowdBT_accs)}, Weighted Accuracy: {np.mean(CrowdBT_waccs)} ± {np.std(CrowdBT_waccs)}, Kendall's Tau: {np.mean(CrowdBT_taus)} ± {np.std(CrowdBT_taus)}")

CrowdBT -- Accuracy: 0.7917298078536987 ± 0.0, Weighted Accuracy: 0.8730530738830566 ± 0.0, Kendall's Tau: 0.5786890421159417 ± 0.0


### FactorBT

In [45]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [46]:
print(device)
gt_scores = to_numpy(df_faceage['score'].tolist())

cuda


In [47]:
from collections import defaultdict
from pathlib import Path

def build_pc_dict_for_noisybt(df, df_faceage, worker_col="worker", left_col="left", right_col="right", label_col="label"):
    """
    Builds the dict NoisyBT_3_0 expects: {worker_id: [(left, right, winner), ...]}

    Mirrors the exact item/worker id mapping used to build PC_faceage_spm:
      - worker ids: position of first appearance in df[worker_col].unique()
      - item ids: position of df_faceage['full_path'] (matched via Path(...).name,
        since df['left']/df['right']/df['label'] are full URLs but full_path
        entries are basenames)

    Unlike PC_faceage_spm (which collapses each row to (winner, loser)), this
    keeps the left/right slot assignment, since NoisyBT's bias parameter needs
    to know which item was shown in the "left" position.

    Returns the dict AND the two id mappings, so you can invert them later
    if needed (e.g. to map scores back to filenames).
    """
    unique_performers = list(df[worker_col].unique())
    performer_label_dict = {performer: i for i, performer in enumerate(unique_performers)}

    item_labels = list(df_faceage["full_path"])
    item_label_dict = {item: i for i, item in enumerate(item_labels)}

    pc_dict = defaultdict(list)
    for performer, group in df.groupby(worker_col):
        key = performer_label_dict[performer]
        for _, row in group.iterrows():
            left = Path(row[left_col]).name
            right = Path(row[right_col]).name
            winner = Path(row[label_col]).name

            left_id = item_label_dict[left]
            right_id = item_label_dict[right]
            winner_id = item_label_dict[winner]

            pc_dict[key].append((left_id, right_id, winner_id))

    return dict(pc_dict), performer_label_dict, item_label_dict

In [48]:
def sort_df(df, column_name):
    # Sort by a specific column (replace 'column_name' with your column)
    df_sorted = df.sort_values(by=column_name, ascending=True)  # or ascending=False

    return df_sorted

df = pd.read_csv('data/crowd_labels.csv')
df = df.rename(columns={'performer': 'worker'})

In [49]:
df.head()

,left,right,label,worker
0,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,0
1,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,0
2,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,0
3,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,1
4,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,1


In [50]:
gt_df = df_faceage
gt_df.head()

,full_path,score,gender
0,nm1442940_rm3965098752_1996-10-3_2006.jpg,10,0.0
1,nm4832920_rm1781768448_2003-8-28_2013.jpg,10,0.0
2,nm0652089_rm860657920_1992-3-10_2002.jpg,10,0.0
3,nm0004917_rm1493730304_1969-5-12_1979.jpg,10,0.0
4,nm1113550_rm1332711936_1996-4-14_2006.jpg,10,0.0


In [51]:
import time
NoisyBT_accs, NoisyBT_waccs, NoisyBT_taus = [], [], []
K = num_reviewers
gt_df = df_faceage
noisybt_time = []

# Build the (left, right, winner) dict NoisyBT_3_0 needs, from the same
# crowd_labels.csv-derived df used by the original NoisyBradleyTerry.
# `df` must have integer-coded 'worker', 'left', 'right', 'label' columns
# aligned with `size`/num_items and gt_df['score'] (see build_pc_dict.py).
PC_faceage_noisybt, _performer_label_dict, _item_label_dict = build_pc_dict_for_noisybt(df, df_faceage)

for seed in range(1):
    try:
        start = time.time()
        noisybt_test = opt_fair.NoisyBT_3_0(data=PC_faceage_noisybt, penalty=0, device=device, random_seed=seed)
        noisybt_scores, noisybt_skills, noisybt_biases = noisybt_test.alternate_optim(size, K, lr_x=lr, lr_y=lr, tol=tol, iters=max_iter)
        end = time.time()
        noisybt_time.append(end-start)
        r_est_np = to_numpy(noisybt_scores)
        gt_scores = to_numpy(gt_df['score'].tolist())
        current_tau = safe_kendalltau(r_est_np, gt_scores)
        if current_tau < 0:
            r_est_np = -r_est_np
        noisybt_acc = compute_acc(gt_df, 1*r_est_np, device)
        noisybt_wacc = compute_weighted_acc(gt_df, 1*r_est_np, device)
        noisybt_tau = safe_kendalltau(r_est_np, gt_scores)
        NoisyBT_accs.append(noisybt_acc)
        NoisyBT_waccs.append(noisybt_wacc)
        NoisyBT_taus.append(noisybt_tau)
    except Exception as e:
        print(e)
        NoisyBT_accs.append(0.0)
        NoisyBT_waccs.append(0.0)
        NoisyBT_taus.append(0.0)

 84%|████████████████████████████████████████████████████▏         | 25251/30000 [02:27<00:27, 170.69it/s, loss=6.27e+4]


In [52]:
print(f"FactorBT -- Accuracy: {np.mean(NoisyBT_accs)} ± {np.std(NoisyBT_accs)}, Weighted Accuracy: {np.mean(NoisyBT_waccs)} ± {np.std(NoisyBT_waccs)}, Kendall's Tau: {np.mean(NoisyBT_taus)} ± {np.std(NoisyBT_taus)}")

FactorBT -- Accuracy: 0.7916303873062134 ± 0.0, Weighted Accuracy: 0.8729999661445618 ± 0.0, Kendall's Tau: 0.578491747529307 ± 0.0
